In [1]:
import pandas as pd
from collections import OrderedDict
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

In [3]:
df=pd.read_excel(r"C:\Users\saurabh kumar\Desktop\Institute\DE\Car Showroom Data.xlsx")

In [5]:
df.head()

,TXNID,County,Media,Time in Mins,Revisits,Probability,Amount
0,1,North America,Application,40,8,36.0,13440.0
1,2,North America,Application,24,4,21.6,7038.0
2,3,North America,Website,31,3,26.8,9875.0
3,4,North America,Application,40,8,36.0,13440.0
4,5,North America,In Store,19,4,11.2,6845.0


In [7]:
df.tail()

,TXNID,County,Media,Time in Mins,Revisits,Probability,Amount
43195,43196,China,Website,18,2,22.0,4054.0
43196,43197,China,Application,32,3,25.4,7094.0
43197,43198,China,Website,8,5,10.0,3648.0
43198,43199,China,Website,29,4,22.6,6627.0
43199,43200,China,Website,29,4,22.6,6627.0


In [9]:
df.describe()

,TXNID,Time in Mins,Revisits,Probability,Amount
count,43200.000000,43200.000000,43200.000000,43200.000000,43200.000000
mean,21600.500000,23.740000,4.820000,25.620000,6812.820000
std,12470.910151,8.267647,2.016853,12.004888,3201.905731
min,1.000000,8.000000,2.000000,8.600000,1784.000000
25%,10800.750000,18.000000,3.000000,17.000000,4473.000000
50%,21600.500000,22.500000,4.500000,22.800000,6215.000000
75%,32400.250000,29.000000,6.000000,30.200000,8412.000000
max,43200.000000,46.000000,10.000000,65.800000,15851.000000


In [11]:
df.dtypes

TXNID             int64
County           object
Media            object
Time in Mins      int64
Revisits          int64
Probability     float64
Amount          float64
dtype: object

In [13]:
#custom summary
def custom_summary(data):
  result=[]
  for col in data.columns:
    if data[col].dtype!="object":
      stats = OrderedDict({
        "column_name":col,
        "count":data[col].count(),
        "minimum":round(data[col].min(),2),
        "quartile_1":round(data[col].quantile(0.25),2),
        "mean":round(data[col].mean(),2),
        "median":round(data[col].median(),2),
        "quartile_3":round(data[col].quantile(0.75),2),
        "maximum":round(data[col].max(),2),
        "variance":round(data[col].var(),2),
        "standard_deviation":round(data[col].std(),2),
        "skewness":round(data[col].skew(),2),
        "kurtosis":round(data[col].kurtosis(),2),
        "IQR":round(data[col].quantile(0.75)-data[col].quantile(0.25),2)})
      result.append(stats)
      if data[col].skew()< -1:
        sklabel="highly negatively skewed"
      elif data[col].skew()>= -1 and data[col].skew()<= -0.5:
        sklabel="moderately negatively skewed"
      elif data[col].skew()> -0.5 and data[col].skew()<= 0:
        sklabel="fairly negatively skewed"
      elif data[col].skew()> 0 and data[col].skew()<= 0.5:
        sklabel="fairly positively skewed"
      elif data[col].skew()> 0.5 and data[col].skew()<= 1:
        sklabel="moderately positively skewed"
      elif data[col].skew()> 1:
        sklabel="highly positively skewed"
      stats["skewness_label"]=sklabel
      if data[col].kurtosis()<-1:
        kurlabel="highlykurtic"
      elif data[col].kurtosis()>= -1 and data[col].kurtosis()<= -0.5:
        kurlabel="moderately kurtic"
      elif data[col].kurtosis()> -0.5 and data[col].kurtosis()<= 0.5:
        kurlabel="mesokurtic"
      elif data[col].kurtosis()> 0.5 and data[col].kurtosis()<= 1:
        kurlabel="moderately leptokurtic"
      elif data[col].kurtosis()> 1:
        kurlabel="highly leptokurtic"
      stats["kurtosis_label"]=kurlabel
  return pd.DataFrame(result)

In [15]:
custom_summary(df)

,column_name,count,minimum,quartile_1,mean,median,quartile_3,maximum,variance,standard_deviation,skewness,kurtosis,IQR,skewness_label,kurtosis_label
0,TXNID,43200,1.0,10800.75,21600.50,21600.5,32400.25,43200.0,1.555236e+08,12470.91,0.00,-1.20,21599.5,fairly negatively skewed,highlykurtic
1,Time in Mins,43200,8.0,18.00,23.74,22.5,29.00,46.0,6.835000e+01,8.27,0.68,0.03,11.0,moderately positively skewed,mesokurtic
2,Revisits,43200,2.0,3.00,4.82,4.5,6.00,10.0,4.070000e+00,2.02,0.63,0.03,3.0,moderately positively skewed,mesokurtic
3,Probability,43200,8.6,17.00,25.62,22.8,30.20,65.8,1.441200e+02,12.00,1.41,2.05,13.2,highly positively skewed,highly leptokurtic
4,Amount,43200,1784.0,4473.00,6812.82,6215.0,8412.00,15851.0,1.025220e+07,3201.91,1.02,0.67,3939.0,highly positively skewed,moderately leptokurtic


In [17]:
#Count
# Minimum
# Quartile 1
# Mean
# Median
# Mode
# Quartile3
# Maximum
# Varinace
# Stad Dev
# Skewness
# kurtosis


In [19]:
df.drop("TXNID",axis=1,inplace=True)

In [21]:
def categorical_encoding(data):
  for col in data.columns:
    if data[col].dtype=="object":
      label_encoder=LabelEncoder()
      data[col]=label_encoder.fit_transform(data[col])
  return data


In [23]:
categorical_encoding(df)

,County,Media,Time in Mins,Revisits,Probability,Amount
0,4,0,40,8,36.0,13440.0
1,4,0,24,4,21.6,7038.0
2,4,2,31,3,26.8,9875.0
3,4,0,40,8,36.0,13440.0
4,4,1,19,4,11.2,6845.0
...,...,...,...,...,...,...
43195,1,2,18,2,22.0,4054.0
43196,1,0,32,3,25.4,7094.0
43197,1,2,8,5,10.0,3648.0
43198,1,2,29,4,22.6,6627.0


In [25]:
def predictive_column(data,target):
  X=data.drop(target,axis=1)
  y=data[target]
  return X,y

In [27]:
x,y=predictive_column(df,"Amount")

In [29]:
def train_and_test_split(x,y,test_size=0.2,random_state=42):
  x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=test_size,random_state=random_state)
  return x_train,x_test,y_train,y_test

In [31]:
x_train,x_test,y_train,y_test = train_and_test_split(x,y)

In [33]:
def build_model(model_name,estimator,x_train,y_train,x_test,y_test):
  model=estimator
  model.fit(x_train,y_train)
  train_score=round(model.score(x_train,y_train),2)
  test_score=round(model.score(x_test,y_test),2)
  return[model_name,train_score,test_score]

In [35]:
build_model("linearregression",LinearRegression(),x_train,y_train,x_test,y_test)

['linearregression', 0.89, 0.88]

In [37]:
def all_models(x_train,y_train,x_test,y_test):
  result=pd.DataFrame(columns=["model_name","train_score","test_score"])
  result.loc[len(result)] = build_model("linearregression",LinearRegression(),x_train,y_train,x_test,y_test)
  result.loc[len(result)] = build_model("ridge",Ridge(),x_train,y_train,x_test,y_test)
  result.loc[len(result)] = build_model("lasso",Lasso(),x_train,y_train,x_test,y_test)
  result.loc[len(result)] = build_model("decisiontreeregressor",DecisionTreeRegressor(),x_train,y_train,x_test,y_test)
  result.loc[len(result)] = build_model("randomforestregressor",RandomForestRegressor(),x_train,y_train,x_test,y_test)
  result.loc[len(result)] = build_model("svr",SVR(),x_train,y_train,x_test,y_test)
  result.loc[len(result)] = build_model("kneighborsregressor",KNeighborsRegressor(),x_train,y_train,x_test,y_test)
  return result.sort_values(by="test_score",ascending=False)

In [39]:
all_models(x_train,y_train,x_test,y_test)

,model_name,train_score,test_score
3,decisiontreeregressor,1.00,1.00
4,randomforestregressor,1.00,1.00
6,kneighborsregressor,1.00,1.00
0,linearregression,0.89,0.88
1,ridge,0.89,0.88
2,lasso,0.89,0.88
5,svr,0.73,0.73


In [ ]:
#decision tree overfit
#random forest is expensive